In [ ]:
%load_ext autoreload
%autoreload 2

import os

os.environ["CUDA_VISIBLE_DEVICES"] = "2"
 
print(os.getcwd())
project_root = os.getcwd()
while not os.path.exists(os.path.join(project_root, "pyproject.toml")) and project_root != os.path.dirname(project_root):
    project_root = os.path.dirname(project_root)
os.chdir(project_root)
print(os.getcwd())

In [ ]:
import json
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick 
import numpy as np
from model_utils.model_config import get_model_path
from transformers import AutoTokenizer, AutoModelForCausalLM

In [ ]:
model_name_list = ["qwen3-4b", "qwen3-8b", "qwen3-14b" ,"toolace-2.5-8b", "watt-tool-8b"]

model_dsiplay_name_dict = {
    "qwen3-4b": "Qwen3-4B",
    "qwen3-8b": "Qwen3-8B",
    "qwen3-14b": "Qwen3-14B",
    "toolace-2.5-8b": "ToolACE-2.5-8B",
    "watt-tool-8b": "Watt-Tool-8B"
} 

add_param_num=0

In [ ]:
result = {}
for model_name in model_name_list:
    result[model_dsiplay_name_dict[model_name]] = {}

    file_dir = os.path.join("data", model_name)

    if model_name in ["toolace-2.5-8b", "watt-tool-8b"]:
        tokenizer = AutoTokenizer.from_pretrained(get_model_path(model_name))

    def check_tool_call(item):
        if model_name not in ["toolace-2.5-8b", "watt-tool-8b"]:
            return 1 if item["logits_info"]["tool_call_token_rank"]==0 else 0
        else:
            out_str = tokenizer.decode(item["logits_info"]["top_token_ids"][0], skip_special_tokens=True)
            if "]" not in out_str:
                if out_str.startswith("["):
                    if len(out_str) == 1:
                        return 1
                    elif out_str[1] == item["tool_name"][0]:
                        return 1
            return 0

    pair_file_path = os.path.join(file_dir, f"pair_add_{add_param_num}.json")
    with open(pair_file_path, "r", encoding="utf-8") as f:
        pair_list = json.load(f)
    pair_list = [check_tool_call(item) for item in pair_list]
    
    random_file_path = os.path.join(file_dir, f"random_add_{add_param_num}.json")
    with open(random_file_path, "r", encoding="utf-8") as f:
        random_list = json.load(f)
    random_list = [check_tool_call(item) for item in random_list]

    gt_file_path = os.path.join(file_dir, f"ground_truth_add_{add_param_num}.json")
    with open(gt_file_path, "r", encoding="utf-8") as f:
        gt_list = json.load(f)
    gt_list = [check_tool_call(item) for item in gt_list]

    result[model_dsiplay_name_dict[model_name]]["Random"] = random_list
    result[model_dsiplay_name_dict[model_name]]["SABEval"] = pair_list
    result[model_dsiplay_name_dict[model_name]]["Relevant"] = gt_list

In [ ]:

import matplotlib.pyplot as plt
import numpy as np

def plot_final_paper_version(result_data):
    """
    Bar chart (Random baseline vs SABEval D_0).
    
    
    
    
    
    """
    
    # Global font setup
    plt.rcParams['font.family'] = 'sans-serif'
    plt.rcParams['font.sans-serif'] = ['Arial', 'Helvetica', 'DejaVu Sans']
    plt.rcParams['font.size'] = 12
    plt.rcParams['axes.linewidth'] = 1.2
    
    # Aggregate means/errs
    models = list(result_data.keys())
    x = np.arange(len(models))
    width = 0.35 
    
    means_random, errs_random = [], []
    means_sab, errs_sab = [], []
    
    max_height = 0 
    
    for model in models:
        r_list = result_data[model]["Random"]
        s_list = result_data[model]["SABEval"]
        
        
        if len(r_list) > 0:
            val = np.mean(r_list) * 100
            err = (np.std(r_list, ddof=1) / np.sqrt(len(r_list))) * 100
        else:
            val, err = 0, 0
        means_random.append(val)
        errs_random.append(err)
        
        # SABEval
        if len(s_list) > 0:
            val_s = np.mean(s_list) * 100
            err_s = (np.std(s_list, ddof=1) / np.sqrt(len(s_list))) * 100
        else:
            val_s, err_s = 0, 0
        means_sab.append(val_s)
        errs_sab.append(err_s)
        
        
        current_max = max(val + err, val_s + err_s)
        if current_max > max_height:
            max_height = current_max

    # Plot
    fig, ax = plt.subplots(figsize=(7, 4), dpi=300)
    
    
    
    # Random baseline
    rects1 = ax.bar(x - width/2, means_random, width, 
                    # yerr=errs_random, 
                    label='Random', 
                    color='#F6CAE5', edgecolor='black', 
                    # hatch='////',
                    linewidth=1,
                    capsize=0) 
    
    # 2. SABEval
    rects2 = ax.bar(x + width/2, means_sab, width, 
                    # yerr=errs_sab, 
                    label='SABEval', 
                    color='#A1A9D0', edgecolor='black', 
                    linewidth=1, alpha=1.0,
                    capsize=0) 

    # Value annotations
    def autolabel(rects):
        for rect in rects:
            height = rect.get_height()
            
            
            if 0 < height < 0.01:
                label_text = '< 0.01'
            elif height == 0:
                label_text = '0.00'
            else:
                label_text = f'{height:.2f}'
            

            offset = max_height * 0.02
            y_pos = height + offset if height > 0 else offset
            
            ax.annotate(label_text,
                        xy=(rect.get_x() + rect.get_width() / 2, height),
                        xytext=(0, 3),
                        textcoords="offset points",
                        ha='center', va='bottom', fontsize=11)

    autolabel(rects1)
    autolabel(rects2)

    # Axes styling
    
    top_limit = max_height * 1.1 if max_height > 0 else 10
    if top_limit > 100: top_limit = 110
    ax.set_ylim(0, top_limit)
    
    
    ax.set_ylabel('Tool Invocation Rate (%)', fontsize=12, fontweight='bold')
    
    
    ax.set_xticks(x)
    ax.set_xticklabels(models, rotation=15, fontsize=12)
    
    
    ax.legend(loc='upper left', frameon=False, fontsize=12)
    
    
    ax.yaxis.grid(True, linestyle='--', which='major', color='grey', alpha=0.3)
    ax.set_axisbelow(True)

    plt.tight_layout()

    plt.savefig("figs/behavior/performance.pdf", bbox_inches="tight")
    plt.show()


plot_final_paper_version(result)